# Hadoop Local Cluster — 1-Hour Class Notebook

**Instructor notebook** for teaching the Docker-based Hadoop cluster in `hadoop-local-docker/`.

| | |
|---|---|
| **Duration** | 60 minutes |
| **Audience** | Data engineering students (beginner–intermediate) |
| **Prerequisite** | Cluster running — see [HADOOP-STUDENT-GUIDE.md](./HADOOP-STUDENT-GUIDE.md) |
| **Scenario** | **ShopStream** — a fictional e-commerce company processing orders, reviews, and clickstream data |

---

## Class agenda (60 min)

| Time | Topic | What students see |
|------|-------|-------------------|
| 0–8 min | Why Hadoop for e-commerce | Story + architecture |
| 8–12 min | Cluster health check | 7 containers healthy |
| 12–22 min | **HDFS** — NameNode + DataNodes | Upload `orders.csv` to HDFS |
| 22–32 min | **YARN** — ResourceManager + workers | `yarn node -list`, run a job |
| 32–45 min | **MapReduce demo** | Word count on product reviews |
| 45–52 min | History Server + Proxy Server | Job logs in UI |
| 52–60 min | E-commerce batch pipeline + AWS mapping | Q&A |

> **Before class:** Run `./scripts/start.sh` (Mac/Linux) or `.\scripts\start.ps1` (Windows PowerShell) and open the web UIs in browser tabs.

---
## Part 1 — The e-commerce story (0–8 min)

### ShopStream's problem

Imagine **ShopStream**, an online store like Amazon or Flipkart. Every day they generate:

| Data type | Example | Size (real world) |
|-----------|---------|-------------------|
| **Orders** | `orders.csv` — who bought what, when | Millions of rows/day |
| **Product reviews** | Free-text feedback | TB of text over time |
| **Clickstream** | Page views, add-to-cart, checkout | Billions of events/day |

They need to:
1. **Store** huge files reliably (not lose data if a server dies)
2. **Process** batches overnight — top products, sentiment trends, fraud checks
3. **Schedule** compute across many machines

**Hadoop** solves this with two core systems:

| Hadoop component | E-commerce analogy |
|------------------|-------------------|
| **HDFS** | Warehouse shelves — store raw order files across multiple racks |
| **YARN** | Shift manager — assigns work to available workers |
| **MapReduce** | Assembly line — many workers each handle a slice of data, combine results |

### Our local cluster = mini ShopStream data center

We run **7 Docker containers** that mimic a real cluster on your laptop.

### Architecture — map every container to ShopStream

```
                    ┌─────────────────────────────────────────┐
                    │           SHOPSTREAM BATCH LAYER         │
                    └─────────────────────────────────────────┘

   HDFS (Storage)                         YARN (Compute Scheduling)
   ──────────────                         ─────────────────────────

   ┌──────────────┐                       ┌──────────────────┐
   │   namenode   │  "Catalog desk"     │ resourcemanager  │  "Shift manager"
   │   :9870 UI   │  knows where every    │   :8088 UI       │  assigns jobs to
   └──────┬───────┘  file block lives     └────────┬─────────┘  worker nodes
          │                                         │
    ┌─────┼─────┬─────────────┐              ┌──────┼──────┬──────────┐
    │     │     │             │              │      │      │          │
 worker-1 worker-2 worker-3                 same 3 workers also run
 (DataNode + NodeManager)                   NodeManager processes
 "Warehouse racks"                          "Packers & analysts"

   Support services
   ────────────────
   historyserver :19888  →  Job archive / audit ("completed shift reports")
   proxyserver   :9099   →  Secure gateway to running apps
```

| Container | Hadoop role | ShopStream real-life role |
|-----------|-------------|---------------------------|
| `namenode` | HDFS NameNode | **Inventory catalog** — tracks which rack holds which order-file chunk |
| `worker-1/2/3` | DataNode | **Warehouse racks** — physically store order CSV blocks (replicated) |
| `worker-1/2/3` | NodeManager | **Analyst desks** — CPU/memory to run batch jobs |
| `resourcemanager` | YARN RM | **Operations manager** — decides which desk runs the "top sellers" report |
| `historyserver` | MR History | **HR archive** — logs of finished jobs for debugging & compliance |
| `proxyserver` | YARN Proxy | **Reception desk** — routes you to the right running job UI |

---
## Part 2 — Pre-flight check (8–12 min)

**Say to class:** *"Before ShopStream opens for business, we verify every department is online."*

Open these URLs in your browser (keep tabs ready):

| UI | URL | Look for |
|----|-----|----------|
| NameNode | http://localhost:9870 | **Live Nodes: 3** |
| YARN | http://localhost:8088 | **Active Nodes: 3** |
| History | http://localhost:19888 | History server home |

In [1]:
# Run from hadoop-local-docker/ folder (or adjust path)
!docker compose ps

NAME              IMAGE                          COMMAND                  SERVICE           CREATED      STATUS                PORTS
historyserver     neshkeev/hadoop:3.3.6-jdk-11   "/tini -- /bin/entry…"   historyserver     7 days ago   Up 7 days (healthy)   0.0.0.0:19888->19888/tcp, [::]:19888->19888/tcp
namenode          neshkeev/hadoop:3.3.6-jdk-11   "/tini -- /bin/entry…"   namenode          7 days ago   Up 7 days (healthy)   0.0.0.0:9870->9870/tcp, [::]:9870->9870/tcp, 0.0.0.0:9900->9000/tcp, [::]:9900->9000/tcp
proxyserver       neshkeev/hadoop:3.3.6-jdk-11   "/tini -- /bin/entry…"   proxyserver       7 days ago   Up 7 days (healthy)   0.0.0.0:9099->9099/tcp, [::]:9099->9099/tcp
resourcemanager   neshkeev/hadoop:3.3.6-jdk-11   "/tini -- /bin/entry…"   resourcemanager   7 days ago   Up 7 days (healthy)   0.0.0.0:8088->8088/tcp, [::]:8088->8088/tcp
worker-1          neshkeev/hadoop:3.3.6-jdk-11   "/tini -- /bin/entry…"   worker-1          7 days ago   Up 7 days (healthy)   0.0.0.0

In [2]:
# Optional: run the verify script instead
!./scripts/verify.sh

Container status:
NAME              IMAGE                          COMMAND                  SERVICE           CREATED      STATUS                PORTS
historyserver     neshkeev/hadoop:3.3.6-jdk-11   "/tini -- /bin/entry…"   historyserver     7 days ago   Up 7 days (healthy)   0.0.0.0:19888->19888/tcp, [::]:19888->19888/tcp
namenode          neshkeev/hadoop:3.3.6-jdk-11   "/tini -- /bin/entry…"   namenode          7 days ago   Up 7 days (healthy)   0.0.0.0:9870->9870/tcp, [::]:9870->9870/tcp, 0.0.0.0:9900->9000/tcp, [::]:9900->9000/tcp
proxyserver       neshkeev/hadoop:3.3.6-jdk-11   "/tini -- /bin/entry…"   proxyserver       7 days ago   Up 7 days (healthy)   0.0.0.0:9099->9099/tcp, [::]:9099->9099/tcp
resourcemanager   neshkeev/hadoop:3.3.6-jdk-11   "/tini -- /bin/entry…"   resourcemanager   7 days ago   Up 7 days (healthy)   0.0.0.0:8088->8088/tcp, [::]:8088->8088/tcp
worker-1          neshkeev/hadoop:3.3.6-jdk-11   "/tini -- /bin/entry…"   worker-1          7 days ago   Up 7 days (

**Expected:** All 7 containers `(healthy)`.

If not healthy → `./scripts/start.sh` and wait 2–3 minutes.

---
## Part 3 — HDFS demo: storing ShopStream orders (12–22 min)

### 3a. NameNode — the catalog desk (5 min)

**Concept:** HDFS splits large files into **blocks** (default 128 MB) and stores **replicas** on DataNodes.

**E-commerce example:** A 10 GB `orders-2026-01.csv` file is split into ~80 blocks. Each block is copied to 2–3 warehouse racks so one rack failure doesn't lose orders.

**Demo:** Open http://localhost:9870 → **Datanodes** tab → confirm 3 live nodes.

In [3]:
# Create ShopStream's HDFS directory structure (like S3 prefixes / data lake zones)
!docker exec namenode hdfs dfs -mkdir -p /shopstream/raw/orders
!docker exec namenode hdfs dfs -mkdir -p /shopstream/raw/reviews
!docker exec namenode hdfs dfs -mkdir -p /shopstream/raw/clickstream
!docker exec namenode hdfs dfs -mkdir -p /shopstream/processed

# Show the directory tree
!docker exec namenode hdfs dfs -ls -R /shopstream

2026-07-16 10:24:01,057 WARN util.NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
2026-07-16 10:24:02,001 WARN util.NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
2026-07-16 10:24:02,949 WARN util.NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
2026-07-16 10:24:03,863 WARN util.NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
2026-07-16 10:24:04,782 WARN util.NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
drwxr-xr-x   - hadoop supergroup          0 2026-07-16 10:16 /shopstream/processed
drwxr-xr-x   - hadoop supergroup          0 2026-07-16 10:16 /shopstream/processed/delivery_reviews
-rw-r--r--   2 hadoop supergroup          0 202

In [4]:
# Upload e-commerce sample files from laptop → HDFS
# (docker cp to container, then hdfs put — works on Mac, Linux, Windows)

!docker cp data/ecommerce/orders.csv namenode:/tmp/orders.csv
!docker cp data/ecommerce/product_reviews.txt namenode:/tmp/product_reviews.txt
!docker cp data/ecommerce/clickstream.csv namenode:/tmp/clickstream.csv

!docker exec namenode hdfs dfs -put -f /tmp/orders.csv /shopstream/raw/orders/orders.csv
!docker exec namenode hdfs dfs -put -f /tmp/product_reviews.txt /shopstream/raw/reviews/product_reviews.txt
!docker exec namenode hdfs dfs -put -f /tmp/clickstream.csv /shopstream/raw/clickstream/clickstream.csv

print("Files uploaded to HDFS")

Successfully copied 3.07kB to namenode:/tmp/orders.csv
Successfully copied 2.56kB to namenode:/tmp/product_reviews.txt
Successfully copied 3.07kB to namenode:/tmp/clickstream.csv
2026-07-16 10:24:06,543 WARN util.NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
2026-07-16 10:24:07,520 WARN util.NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
2026-07-16 10:24:08,496 WARN util.NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Files uploaded to HDFS


In [5]:
# Prove the data is in HDFS — peek at ShopStream orders
!docker exec namenode hdfs dfs -ls /shopstream/raw/orders
!echo "--- First 5 order rows ---"
!docker exec namenode hdfs dfs -cat /shopstream/raw/orders/orders.csv | head -6

2026-07-16 10:24:09,509 WARN util.NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Found 1 items
-rw-r--r--   2 hadoop supergroup       1254 2026-07-16 10:24 /shopstream/raw/orders/orders.csv
--- First 5 order rows ---
2026-07-16 10:24:10,725 WARN util.NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
order_id,customer_id,product_id,product_name,category,quantity,unit_price,order_date,status,city
10001,C1001,P501,Wireless Headphones,Electronics,1,79.99,2026-01-05,Delivered,Mumbai
10002,C1002,P302,Cotton T-Shirt,Apparel,2,19.99,2026-01-05,Delivered,Delhi
10003,C1003,P501,Wireless Headphones,Electronics,1,79.99,2026-01-06,Shipped,Bangalore
10004,C1001,P210,Running Shoes,Footwear,1,129.99,2026-01-06,Delivered,Mumbai
10005,C1004,P880,Blender,Home,1,49.99,2026-01-07,Delivered,Chennai


### 3b. DataNodes — warehouse racks (5 min)

**Say to class:** *"The NameNode doesn't store the actual order rows — workers do."*

**Real life:** Like Amazon S3 — metadata in one place, bytes spread across storage nodes.

**Demo:** In NameNode UI → **Utilities → Browse the file system** → navigate to `/shopstream/raw/orders/orders.csv` → see block locations on `worker-1`, `worker-2`, `worker-3`.

In [6]:
# CLI: show file size and replication factor
!docker exec namenode hdfs dfs -stat "%n | size=%b bytes | replication=%r" /shopstream/raw/orders/orders.csv

2026-07-16 10:24:11,707 WARN util.NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
orders.csv | size=1254 bytes | replication=2


**Talking point:** Our cluster sets replication **= 2**. Real e-commerce production often uses **3** across availability zones.

---
## Part 4 — YARN demo: scheduling batch work (22–32 min)

### ResourceManager — the shift manager

**E-commerce example:** At 2 AM, ShopStream runs:
- Job A: Calculate top 10 products by revenue
- Job B: Flag fraudulent orders
- Job C: Aggregate clickstream funnels

YARN decides **which worker** gets **how much CPU/RAM** for each job.

**Demo:** Open http://localhost:8088 → **Nodes** tab.

In [7]:
# List YARN worker nodes (our 3 Docker workers)
!docker exec resourcemanager yarn node -list

2026-07-16 10:24:12,600 WARN util.NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
2026-07-16 10:24:12,627 INFO client.DefaultNoHARMFailoverProxyProvider: Connecting to ResourceManager at resourcemanager/172.22.0.3:8032
Total Nodes:3
         Node-Id	     Node-State	Node-Http-Address	Number-of-Running-Containers
  worker-2:39883	        RUNNING	   worker-2:28042	                           0
  worker-3:34491	        RUNNING	   worker-3:38042	                           0
  worker-1:35421	        RUNNING	   worker-1:18042	                           0


In [8]:
# Show cluster resource capacity
!docker exec resourcemanager yarn node -list -all

2026-07-16 10:24:13,396 WARN util.NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
2026-07-16 10:24:13,425 INFO client.DefaultNoHARMFailoverProxyProvider: Connecting to ResourceManager at resourcemanager/172.22.0.3:8032
Total Nodes:3
         Node-Id	     Node-State	Node-Http-Address	Number-of-Running-Containers
  worker-2:39883	        RUNNING	   worker-2:28042	                           0
  worker-3:34491	        RUNNING	   worker-3:38042	                           0
  worker-1:35421	        RUNNING	   worker-1:18042	                           0


**Map containers to e-commerce:**

| YARN term | ShopStream meaning |
|-----------|-------------------|
| NodeManager | One warehouse shift crew (worker container) |
| Container (YARN) | Slice of CPU/RAM assigned to one task |
| Application | One batch report (e.g. "daily revenue summary") |
| Queue | Priority lane — e.g. "finance jobs" before "experiments" |

---
## Part 5 — MapReduce demo: analyze product reviews (32–45 min)

### Business question

> *"What words appear most often in ShopStream product reviews?"*

This mimics a **batch sentiment / keyword** pipeline before ML models existed at scale.

### How MapReduce works (whiteboard + say aloud)

1. **Map** — each worker reads some review lines, emits `(word, 1)` pairs
2. **Shuffle** — framework groups same words together
3. **Reduce** — sum counts per word → `delivery: 3`, `excellent: 2`, ...

**E-commerce parallel:** 1000 analysts each count words on 1000 reviews, then one lead merges totals.

In [9]:
# Clean previous output if re-running the demo
!docker exec namenode hdfs dfs -rm -r -f /shopstream/processed/review_wordcount 2>/dev/null || true

# Submit wordcount MapReduce job via YARN
!docker exec namenode bash -lc \
  'hadoop jar $(ls $HADOOP_HOME/share/hadoop/mapreduce/hadoop-mapreduce-examples-*.jar | head -1) wordcount /shopstream/raw/reviews/product_reviews.txt /shopstream/processed/review_wordcount'

Deleted /shopstream/processed/review_wordcount
2026-07-16 10:24:15,126 WARN util.NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
2026-07-16 10:24:15,354 INFO client.DefaultNoHARMFailoverProxyProvider: Connecting to ResourceManager at resourcemanager/172.22.0.3:8032
2026-07-16 10:24:15,472 INFO mapreduce.JobResourceUploader: Disabling Erasure Coding for path: /tmp/hadoop-yarn/staging/hadoop/.staging/job_1783516535069_0007
2026-07-16 10:24:15,569 INFO input.FileInputFormat: Total input files to process : 1
2026-07-16 10:24:15,598 INFO mapreduce.JobSubmitter: number of splits:1
2026-07-16 10:24:15,667 INFO mapreduce.JobSubmitter: Submitting tokens for job: job_1783516535069_0007
2026-07-16 10:24:15,667 INFO mapreduce.JobSubmitter: Executing with tokens: []
2026-07-16 10:24:15,724 INFO conf.Configuration: resource-types.xml not found
2026-07-16 10:24:15,724 INFO resource.ResourceUtils: Unable to find 'resource-types.xm

**While job runs (~15 sec):** Switch to http://localhost:8088/cluster/apps — show app going from `ACCEPTED` → `RUNNING` → `FINISHED`.

In [10]:
# Read the batch report output from HDFS
!docker exec namenode hdfs dfs -cat /shopstream/processed/review_wordcount/part-r-00000 | sort -t$'\t' -k2 -nr | head -15

2026-07-16 10:24:26,611 WARN util.NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
was	3
for	3
customer	3
recommend	2
quality	2
poor	2
packaging	2
is	2
headphones	2
excellent	2
delivery	2
after	2
Blender	2
would	1
works	1


**Discuss results:** Words like `delivery`, `excellent`, `disappointed` — in production you'd feed this into dashboards or NLP pipelines.

### Bonus demo (2 min): grep for negative reviews mentioning "delivery"

MapReduce `grep` example = filter reviews containing a pattern (like SQL `LIKE` at scale).

In [11]:
!docker exec namenode hdfs dfs -rm -r -f /shopstream/processed/delivery_reviews 2>/dev/null || true

!docker exec namenode bash -lc \
  'hadoop jar $(ls $HADOOP_HOME/share/hadoop/mapreduce/hadoop-mapreduce-examples-*.jar | head -1) grep /shopstream/raw/reviews/product_reviews.txt /shopstream/processed/delivery_reviews "delivery.*"'

!docker exec namenode hdfs dfs -cat /shopstream/processed/delivery_reviews/part-m-00000

Deleted /shopstream/processed/delivery_reviews
2026-07-16 10:24:28,510 WARN util.NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
2026-07-16 10:24:28,756 INFO client.DefaultNoHARMFailoverProxyProvider: Connecting to ResourceManager at resourcemanager/172.22.0.3:8032
2026-07-16 10:24:28,881 INFO mapreduce.JobResourceUploader: Disabling Erasure Coding for path: /tmp/hadoop-yarn/staging/hadoop/.staging/job_1783516535069_0008
2026-07-16 10:24:28,975 INFO input.FileInputFormat: Total input files to process : 1
2026-07-16 10:24:29,002 INFO mapreduce.JobSubmitter: number of splits:1
2026-07-16 10:24:29,073 INFO mapreduce.JobSubmitter: Submitting tokens for job: job_1783516535069_0008
2026-07-16 10:24:29,073 INFO mapreduce.JobSubmitter: Executing with tokens: []
2026-07-16 10:24:29,124 INFO conf.Configuration: resource-types.xml not found
2026-07-16 10:24:29,124 INFO resource.ResourceUtils: Unable to find 'resource-types.xm

---
## Part 6 — History Server & Proxy Server (45–52 min)

### historyserver — completed shift reports

**E-commerce use:** Audit trail — *"Who ran the revenue job on Jan 12? How long did it take? Did it fail?"*

**Demo:** Open http://localhost:19888 → find the finished `wordcount` job → click through to counters (records read, map/reduce tasks).

### proxyserver — reception desk for app UIs

When a job runs, YARN gives a tracking URL through the proxy (port **9099**). In production this adds security — users don't connect directly to worker nodes.

**Real life:** Same pattern as Kubernetes ingress or AWS ALB in front of microservices.

In [12]:
# List recent YARN applications from CLI
!docker exec resourcemanager yarn application -list -appStates FINISHED

2026-07-16 10:24:56,434 WARN util.NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
2026-07-16 10:24:56,466 INFO client.DefaultNoHARMFailoverProxyProvider: Connecting to ResourceManager at resourcemanager/172.22.0.3:8032
Total number of applications (application-types: [], states: [FINISHED] and tags: []):9
                Application-Id	    Application-Name	    Application-Type	      User	     Queue	             State	       Final-State	       Progress	                       Tracking-URL
application_1783516535069_0001	          word count	           MAPREDUCE	    hadoop	   default	          FINISHED	         SUCCEEDED	           100%	http://historyserver:19888/jobhistory/job/job_1783516535069_0001
application_1783516535069_0002	          word count	           MAPREDUCE	    hadoop	   default	          FINISHED	         SUCCEEDED	           100%	http://historyserver:19888/jobhistory/job/job_1783516535069_0002
applicat

---
## Part 7 — End-to-end e-commerce batch pipeline (52–60 min)

### How ShopStream would use this in production

```
  Web / Mobile App                Nightly Batch (Hadoop)
  ─────────────────              ───────────────────────
  Orders API  ──►  Kafka/Logs  ──►  HDFS /shopstream/raw/orders/
  Reviews     ──►  Files       ──►  HDFS /shopstream/raw/reviews/
  Clickstream ──►  Events      ──►  HDFS /shopstream/raw/clickstream/
                                        │
                                        ▼
                                 MapReduce / Spark jobs
                                 (YARN schedules on workers)
                                        │
                                        ▼
                                 HDFS /shopstream/processed/
                                 or load to warehouse (Redshift,
                                 Snowflake, BigQuery)
                                        │
                                        ▼
                                 BI Dashboard: top products,
                                 conversion funnel, review trends
```

### Map our local cluster → AWS (for this course)

| Local (today) | AWS equivalent |
|---------------|----------------|
| HDFS | **Amazon S3** (storage) |
| YARN + MapReduce | **Amazon EMR** (managed Hadoop/Spark) |
| NameNode UI | S3 console + Glue Catalog |
| Batch jobs | EMR steps, Glue jobs, or Lambda + S3 triggers |

### Key takeaways for students

1. **HDFS / S3** = durable storage for raw e-commerce data
2. **NameNode / Catalog** = knows *where* data lives
3. **DataNodes / S3 shards** = actual bytes, replicated for fault tolerance
4. **YARN** = scheduler — picks workers for batch jobs
5. **MapReduce** = classic batch pattern (map → shuffle → reduce)
6. **History Server** = job audit & debugging

### Homework ideas

- Upload your own CSV to `/shopstream/raw/` and run wordcount
- In NameNode UI, find block locations for a file
- Watch a job in YARN UI from `RUNNING` to `FINISHED`
- Read [HADOOP-STUDENT-GUIDE.md](./HADOOP-STUDENT-GUIDE.md) for setup on your machine

---
## Instructor appendix

### Quick reference — all services

| # | Container | Port | Demo command / UI |
|---|-----------|------|-------------------|
| 1 | namenode | 9870, 9900 | `hdfs dfs -ls /` · http://localhost:9870 |
| 2 | worker-1 | 18042, 19864 | Blocks visible in NameNode UI |
| 3 | worker-2 | 28042, 29864 | Same |
| 4 | worker-3 | 38042, 39864 | Same |
| 5 | resourcemanager | 8088 | `yarn node -list` · http://localhost:8088 |
| 6 | historyserver | 19888 | http://localhost:19888 |
| 7 | proxyserver | 9099 | Linked from YARN app tracking URL |

### If demo fails mid-class

```bash
docker compose ps
docker compose restart worker-1 worker-2 worker-3
./scripts/smoke-test.sh
```

### Stop cluster after class

```bash
./scripts/stop.sh
```